# 02 - Data Cleaning

Create `data/processed/jira_issues_cleaned.csv` from the raw Jira export. This keeps only fields known before resolution, plus dates needed by the next notebook to create `duration_days`.

In [7]:
from pathlib import Path

import pandas as pd

In [8]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "jira_ticket_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_COLUMNS = [
    "summary",
    "description",
    "priority.name",
    "issuetype.name",
    "project.key",
    "projectCategory.name",
    "votes.votes",
    "watches.watchCount",
    "labels",
    "assignee",
    "statusCategory.name",
    "created",
    "resolutiondate",
]

ticket_df = pd.read_csv(RAW_PATH, usecols=RAW_COLUMNS)

print(f"Raw rows loaded: {ticket_df.shape[0]:,}")
print(f"Raw columns loaded: {ticket_df.shape[1]:,}")

Raw rows loaded: 1,149,323
Raw columns loaded: 13


In [9]:
created_dates = pd.to_datetime(ticket_df["created"], errors="coerce")
resolution_dates = pd.to_datetime(ticket_df["resolutiondate"], errors="coerce")

completed_issue_mask = (
    ticket_df["statusCategory.name"].eq("Done")
    & created_dates.notna()
    & resolution_dates.notna()
    & (resolution_dates >= created_dates)
)

rows_before = len(ticket_df)
clean_df = ticket_df.loc[completed_issue_mask].copy()

print(f"Rows before completed issue filtering: {rows_before:,}")
print(f"Rows after completed issue filtering: {len(clean_df):,}")
print(f"Rows removed: {rows_before - len(clean_df):,}")

Rows before completed issue filtering: 1,149,323
Rows after completed issue filtering: 945,103
Rows removed: 204,220


In [10]:
text_columns = ["summary", "description"]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

categorical_mappings = {
    "priority.name": "priority_name",
    "issuetype.name": "issuetype_name",
    "project.key": "project_key",
    "projectCategory.name": "project_category_name",
}

for source_column, clean_column in categorical_mappings.items():
    clean_df[clean_column] = (
        clean_df[source_column]
        .fillna("Unknown")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", "Unknown")
    )

clean_df["summary_char_count"] = clean_df["summary"].str.len()
clean_df["summary_word_count"] = clean_df["summary"].str.split().str.len()
clean_df["description_char_count"] = clean_df["description"].str.len()
clean_df["description_word_count"] = clean_df["description"].str.split().str.len()
clean_df["has_description"] = (clean_df["description_word_count"] > 0).astype(int)

labels_text = clean_df["labels"].fillna("[]").astype(str).str.strip()
clean_df["labels_count"] = labels_text.str.count(",") + labels_text.ne("[]").astype(int)
clean_df["has_assignee"] = clean_df["assignee"].notna().astype(int)
clean_df["votes_votes"] = pd.to_numeric(clean_df["votes.votes"], errors="coerce").fillna(0).clip(lower=0)
clean_df["watches_watch_count"] = pd.to_numeric(clean_df["watches.watchCount"], errors="coerce").fillna(0).clip(lower=0)

for column in ["created", "resolutiondate"]:
    clean_df[column] = pd.to_datetime(clean_df[column], errors="coerce")

In [11]:
cleaned_model_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "project_key",
    "project_category_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "has_description",
    "labels_count",
    "has_assignee",
    "votes_votes",
    "watches_watch_count",
    "created",
    "resolutiondate",
]

rows_before = len(clean_df)
model_df = clean_df[cleaned_model_columns].drop_duplicates().copy()

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal: {len(model_df):,}")
print(f"Duplicate rows removed: {rows_before - len(model_df):,}")

missing_summary = pd.DataFrame({
    "missing_count": model_df.isna().sum(),
    "missing_percent": model_df.isna().mean().mul(100).round(2),
})
missing_summary.sort_values("missing_percent", ascending=False)

Rows before duplicate removal: 945,103
Rows after duplicate removal: 937,203
Duplicate rows removed: 7,900


,missing_count,missing_percent
summary,0,0.0
description,0,0.0
priority_name,0,0.0
issuetype_name,0,0.0
project_key,0,0.0
project_category_name,0,0.0
summary_char_count,0,0.0
summary_word_count,0,0.0
description_char_count,0,0.0
description_word_count,0,0.0


In [ ]:
csv_path = OUTPUT_DIR / "jira_issues_cleaned.csv"
sample_path = OUTPUT_DIR / "jira_issues_cleaned_sample.csv"

model_df.to_csv(csv_path, index=False)
model_df.sample(n=min(100, len(model_df)), random_state=42).to_csv(sample_path, index=False)

print(f"Saved cleaned CSV file to: {csv_path}")
print(f"Saved sample CSV file to: {sample_path}")
print(f"Final cleaned rows: {model_df.shape[0]:,}")
print(f"Final cleaned columns: {model_df.shape[1]:,}")